## Identify potential extreme rainfall events from GloFAS reforecast data

- Read in data from 2003 to 2023 
- 10 ensemble members per forecast
- Forecasts run every 2-3 days on average 
- Lead time T+0 through to T+240 (10-day)

In [1]:
import glob
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import logging
from pathlib import Path
import time

In [3]:
# logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")

# CONFIG - adjust as needed
ROOT = Path("/mnt/metdata/W25-2348/glofas/ensemble_reforecasts")
OUTPUT_DIR = ROOT / "processed_ensemble_members"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#YEARS = list(range(2014, 2023))   # default years 2003..2023 inclusive
# Or use a custom list e.g. YEARS = [2019, 2022, 2023]
YEARS = [2017]

GRIB_FILENAME = "data.grib"       # file inside each monthly folder (20xx_01/data.grib)
VARIABLE = "dis24"                # the variable name in your files
SPATIAL_DIMS = ["latitude", "longitude"]
ENSEMBLE_DIM = "number"
STEP_DIM = "step"
TIME_DIM = "time"

def find_year_files(year):
    """Return sorted list of monthly GRIB files for given year directory"""
    pattern = str(ROOT / f"{year}" / f"cems-glofas-reforecast_{year}_*" / GRIB_FILENAME)
    files = sorted(glob.glob(pattern))
    return files


def open_year_dataset(files):
    """ 
    Try to open files using 'mfdataset', otherwise open individually and concat 
    """
    if not files:
        raise FileNotFoundError("No files found for year")
    
    try:
        # attempt fast merge
        logging.info("Trying xr.open_mfdataset on %d files ...", len(files))
        ds = xr.open_mfdataset(files, 
                               engine='cfgrib', 
                               combine='by_coords',
                               parallel=True,
                               backend_kwargs={'indexpath': ''})
        return ds
    except Exception as e:
        logging.warning("open_mfdataset failed: %s. Falling back to per-file open+concat.", e)

    # fallback: open each file separately, optionally preprocess, then concat
    datasets = []
    for f in files:
        logging.info("Opening %s", f)
        dsf = xr.open_dataset(f, engine='cfgrib', backend_kwargs={"indexpath": ""})  # per-file open
        datasets.append(dsf)

    # concat on time (ensure each ds has a 'time' coordinate)
    ds = xr.concat(datasets, dim='time', data_vars='minimal', coords='minimal', combine_attrs='override')
    return ds


def compute_spread_for_year(year, out_dir=OUTPUT_DIR, overwrite=False):
    files = find_year_files(year)
    if not files:
        logging.warning("No files for year %s -> skipping", year)
        return None

    outpath = out_dir / f"glofas_spread_{year}.nc"
    if outpath.exists() and not overwrite:
        logging.info("Output exists for %s -> %s (skip). Use overwrite=True to replace.", year, outpath)
        return outpath

    ds = open_year_dataset(files)
    if VARIABLE not in ds:
        raise KeyError(f"Variable {VARIABLE} not found in dataset variables: {list(ds.keys())}")

    da = ds[VARIABLE].sel(latitude=53.375,method='nearest').sel(longitude=353.625,method='nearest')

    # # Spatial average to remove lat/lon
    # if all(dim in da.dims for dim in SPATIAL_DIMS):
    #     da = da.mean(dim=SPATIAL_DIMS)  # dims: (number, time, step)
    # else:
    #     logging.warning("One of spatial dims %s not in da.dims %s; skipping spatial mean",
    #                     SPATIAL_DIMS, da.dims)

    # Ensure dims order we expect (number, time, step)
    # We will use .isel(step=...) which expects step present
    logging.info("After selecting single grid point, dims = %s", da.dims)

    print(da)

    # get dims sizes and coords
    nstep = da.sizes.get(STEP_DIM)
    ntime = da.sizes.get(TIME_DIM)
    if nstep is None or ntime is None:
        raise ValueError("Expected dims 'step' and 'time' in data array after spatial averaging")

    times = da[TIME_DIM].values
    steps = da[STEP_DIM].values

    # Prepare arrays to hold results: shape (time, step)
    q25_arr = np.full((ntime, nstep), np.nan, dtype=np.float32)
    q50_arr = np.full((ntime, nstep), np.nan, dtype=np.float32)
    q75_arr = np.full((ntime, nstep), np.nan, dtype=np.float32)
    spread_maxmin_arr = np.full((ntime, nstep), np.nan, dtype=np.float32)

    logging.info("Computing percentiles for year %s: nstep=%d ntime=%d", year, nstep, ntime)

    # Loop over steps - each step we bring a small (ensemble x time) array into memory
    for si in range(nstep):
        t0 = time.time()
        da_step = da.isel({STEP_DIM: si})  # dims: (number, time)
        # compute to numpy array in memory (small: ensemble ~10, time ~175)
        arr = da_step.data
        if hasattr(arr, "compute"):
            arr = arr.compute()   # returns numpy array
        else:
            arr = np.asarray(arr)

        # shape should be (number, time)
        if arr.ndim != 2:
            # try to squeeze or transpose if needed
            arr = np.squeeze(arr)
            if arr.ndim != 2:
                raise ValueError(f"Unexpected arr shape for step {si}: {arr.shape}")

        # compute percentiles across ensemble axis (axis=0)
        # use nanpercentile to be robust to missing values
        q25 = np.nanpercentile(arr, 25, axis=0)
        q50 = np.nanpercentile(arr, 50, axis=0)
        q75 = np.nanpercentile(arr, 75, axis=0)
        maxv = np.nanmax(arr, axis=0)
        minv = np.nanmin(arr, axis=0)
        spread_mm = maxv - minv

        # store
        q25_arr[:, si] = q25.astype(np.float32)
        q50_arr[:, si] = q50.astype(np.float32)
        q75_arr[:, si] = q75.astype(np.float32)
        spread_maxmin_arr[:, si] = spread_mm.astype(np.float32)

        logging.info("  step %d/%d done in %.2f s", si + 1, nstep, time.time() - t0)

    # Build xarray Dataset for output
    out_ds = xr.Dataset(
        {
            "q25": (("time", "step"), q25_arr),
            "q50": (("time", "step"), q50_arr),
            "q75": (("time", "step"), q75_arr),
            "spread_maxmin": (("time", "step"), spread_maxmin_arr),
        },
        coords={
            "time": times,
            "step": steps,
        },
        attrs={
            "source": "computed from dis24",
            "year": int(year),
        },
    )

    # Save to netcdf
    logging.info("Saving output to %s", outpath)
    encoding = {var: {"zlib": True, "complevel": 4} for var in out_ds.data_vars}
    out_ds.to_netcdf(outpath, format="NETCDF4", encoding=encoding)
    logging.info("Saved %s", outpath)
    return outpath


def main(years=YEARS):
    for y in years:
        try:
            compute_spread_for_year(y)
        except Exception as e:
            logging.exception("Failed processing year %s: %s", y, e)


if __name__ == "__main__":
    main()

2025-11-05 11:51:52,730 INFO: Trying xr.open_mfdataset on 12 files ...
2025-11-05 11:53:00,581 INFO: After selecting single grid point, dims = ('number', 'time', 'step')
2025-11-05 11:53:00,587 INFO: Computing percentiles for year 2017: nstep=9 ntime=175


<xarray.DataArray 'dis24' (number: 10, time: 175, step: 9)> Size: 63kB
dask.array<getitem, shape=(10, 175, 9), dtype=float32, chunksize=(10, 18, 9), chunktype=numpy.ndarray>
Coordinates:
  * number      (number) int64 80B 1 2 3 4 5 6 7 8 9 10
  * time        (time) datetime64[ns] 1kB 2017-01-01 2017-01-04 ... 2017-12-28
  * step        (step) timedelta64[ns] 72B 1 days 2 days ... 8 days 9 days
    surface     float64 8B 0.0
    latitude    float64 8B 53.38
    longitude   float64 8B 353.6
    valid_time  (time, step) datetime64[ns] 13kB dask.array<chunksize=(9, 9), meta=np.ndarray>
Attributes: (12/31)
    GRIB_paramId:                             240024
    GRIB_dataType:                            pf
    GRIB_numberOfPoints:                      3600
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            avg
    ...                                       ...
    GRIB_shortName:                 

KeyboardInterrupt: 